<div style="text-align: center">
<img src="https://github.com/LinkedEarth/Logos/blob/master/PyleoTUPS/pyleotups_logo.png?raw=true" alt="PyleoTUPS logo" width="400">
</div>

# Working with PANGAEA

## Authors

Deborah Khider [![ORCID](https://img.shields.io/badge/ORCID-0000--0001--7501--8430-A6CE39?logo=orcid)](https://orcid.org/0000-0001-7501-8430), Dhiren Oswal [![ORCID](https://img.shields.io/badge/ORCID-0009--0001--2495--2626-A6CE39?logo=orcid)](https://orcid.org/0009-0001-2495-2626)


## Preamble

This tutorial introduces the `PangaeaDataset` object and its functionalities in PyleoTUPS, focusing on searching by unique ID. A separate tutorial covers the full range of PANGAEA search capabilities.

Note that PANGAEA and NOAA work differently under the hood, though PyleoTUPS aims to provide a consistent user experience across both repositories. Most differences arise from the search query parameters.

### Goals

 - Search for datasets using a specific ID
 - Obtain information such as location, publication, variable metadata and the associated data.

### Pre-requisites

- A familiarity with PANGAEA data repository
- A familiarity the [pangaeapy](https://pangaea-data-publisher.github.io/pangaeapy/) package upon which `PyleoTUPS` is built. 

### Reading time

15 min

Let's import our packages!

In [2]:
import pyleotups as pt
import pandas as pd

## Accessing a dataset using the PANGAEA Unique ID

This is the most basic search you can perform, but if you know which dataset you are looking for, it is the easiest way to get all the needed information. First, you need to create a `PangaeaDataset` object that will store the information.

In [3]:
ds = pt.PangaeaDataset()

In this example, we will be looking at the dataset from [Todorovic et al. (2024)](https://doi.org/10.1594/PANGAEA.965772), which is a supplement to [Todorovic et al. (2024)](https://doi.org/10.1029/2024PA004843). From this example, you can see that PANGAEA mints DOI for each of its datasets. When using the data in a publication, a good practice is to cite both the data citation (from PANGAEA) and the original paper (in this case from Paleoceanography and Paleoclimatology).

Let's have a look at the PANGAEA DOI: 0.1594/PANGAEA.965772. The last 6 numbers can be considered a unique ID that you can pass to `PyleoTUPS` much like you would pass a `noaa_id`:

In [4]:
res = ds.search_studies(study_ids = '965772')

[2026-04-24 10:22:38,057][INFO] - Registering Study 965772 via direct lookup.
[2026-04-24 10:22:40,221][INFO] - Retrived 1 studies


<div style="border-left: 4px solid #1f77b4; background: #f0f8ff; padding: 0.5em 1em; border-radius: 4px;">
  <strong>Note:</strong> Unlike The NOAADataset object, the PangaeaDataset object will accept multiple IDs, passed as a list.
</div>

The  `summary` method provides basic information about the dataset, mirroring what is available from NOAA (e.g., time bounds, notes, keywords). However, there are two important differences to be aware of:

* PANGAEA does not store year/age information the same way NOAA does, since PANGAEA serves a much broader community than paleoscience. Time information at the metadata level typically refers to when the archive or data was collected, not the paleoclimate age. PyleoTUPS attempts to retrieve age information directly from the data files via the `AGE` column.
* PANGAEA has no concept of `Site`. For paleoclimate data, the closely related concept of `Event` serves a similar role; `PyleoTUPS` maps it to the NOAA `Site` nomenclature for consistency.

The method returns a `pandas.DataFrame`.

In [5]:
df_summary = ds.get_summary()
display(df_summary)

,StudyID,StudyName,EarliestYearBP,MostRecentYearBP,EarliestYearCE,MostRecentYearCE,StudyNotes,ScienceKeywords,Investigators,Publications,Sites,Funding,CollectionMembers
0,965772,"Paired d18O and Sr/Ca, and reconstructed d18Os...",128,-54,1821,2004,Two massive living Porites sp. coral colonies ...,"[coral climatology, Oxygen isotopes, Pacific W...","Todorovic, Sara, Dissard, Delphine, Linsley, B...","Todorovic, Sara; Dissard, Delphine; Linsley, B...","[Rotuma_RO2, Tonga_TNI2]",[{'name': 'Make Our Planet Great Again-Ocean a...,None


### Obtaining some basic metadata information: geography, bibliography. and funding:

Some of the basic functionalities such as retrieving the geographical, bibliographical, and funding information work the same way on the `PangaeaDataset` object as it did on the `NOAADataset` object:

In [6]:
df_geo = ds.get_geo()
display(df_geo)

,StudyID,SiteID,SiteName,LocationName,Latitude,Longitude,Elevation
0,965772,3120868,Rotuma_RO2,South Pacific Convergence Zone,-12.488056,177.106944,-11.0
1,965772,3120871,Tonga_TNI2,South Pacific Convergence Zone,-20.278056,-174.815278,-3.5


<div style="border-left: 4px solid #1f77b4; background: #f0f8ff; padding: 0.5em 1em; border-radius: 4px;">
  <strong>Note:</strong> The SiteID field is available through pangaeapy and included here for completeness. While conceptually equivalent to the NOAA SiteID, it cannot be used programmatically to access data for a specific site.
</div>

In [7]:
bib, df_pub = ds.get_publications()
display(df_pub)

,StudyID,StudyName,Author,Title,Journal,Year,Volume,Number,Pages,Type,DOI,URL,CitationKey
0,965772,"Paired d18O and Sr/Ca, and reconstructed d18Os...","Todorovic, Sara; Dissard, Delphine; Linsley, B...","Paired d18O and Sr/Ca, and reconstructed d18Os...",PANGAEA,2024,None,None,None,dataset,10.1594/PANGAEA.965772,https://doi.org/10.1594/PANGAEA.965772,"Todorovic, Sara; Dissard, Delphine; Linsley, B..."


In [8]:
df_funding = ds.get_funding()
display(df_funding)

,StudyID,StudyName,FundingAgency,FundingGrant
0,965772,"Paired d18O and Sr/Ca, and reconstructed d18Os...",None,MOPGA-OASIS / 4896


### Get Variable Information

Accessing the data and variable information is where PANGAEA differs from NOAA the most. First, although `Event` is closely related to the concept `Site`, `PyleoTUPS` cannot retrieve the data independently for each site. When PANGAEA creates a Dataset on its repository, the data publisher adds information to the data table pertaining to these sites: the site name under the column header `Event` and geographical coordinates. We will see an example of this as we look at the data. Second, PANGAEA only allows one table per dataset. If a study results in multiple tables, then they are stored under a collection. We will look at an example of a collection in this notebook's next section. Consequently, the methods `get_sites` and `get_tables` are not supported for `PangaeaDataset`. 

Let's have a look at the data: 

In [9]:
df_data = ds.get_data(study_id = '965772')[0]
display(df_data.head())

,Event,Age,Age_2,Date,Porites sp. Sr/Ca,Porites sp. δ18O,δ18Osw,Latitude,Longitude,Elevation,Date/Time
0,Rotuma_RO2,0.128986,1821.013528,1821-01-01,8.946,-4.640,0.690,-12.488056,177.106944,-11.0,1998-10-27
1,Rotuma_RO2,0.128903,1821.096851,1821-02-01,8.904,-4.625,0.756,-12.488056,177.106944,-11.0,1998-10-27
2,Rotuma_RO2,0.128820,1821.180174,1821-03-01,8.898,-4.650,0.772,-12.488056,177.106944,-11.0,1998-10-27
3,Rotuma_RO2,0.128737,1821.263497,1821-04-01,8.910,-4.600,0.788,-12.488056,177.106944,-11.0,1998-10-27
4,Rotuma_RO2,0.128653,1821.346820,1821-05-01,8.915,-4.575,0.804,-12.488056,177.106944,-11.0,1998-10-27


Note that although a dataset will only have one table associated with it, we return a list for consistency with the output of the same functionality for `NOAADataset`. 

As mentioned previously, PANGAEA data table will contain information about the Event and associated geographical coordinate as well as a `Date` which is not useful for most paleoclimate study. Let's have a look at the unique entries in `Event`:

In [10]:
df_data['Event'].unique()

<StringArray>
['Rotuma_RO2', 'Tonga_TNI2']
Length: 2, dtype: str

As you see, data for the two Sites are stored in the same data table. 

To get meatadata information about the variables (i.e., columns), you can use the same `get_variables` function:

In [11]:
df_var = ds.get_variables(study_ids = '965772')
display(df_var)

,StudyID,VariableName,ShortName,Unit,OntologyTerms
0,965772,Event label,Event,NaN,[]
1,965772,AGE,Age,ka BP,"[{'id': 1073135, 'name': 'age', 'semantic_uri'..."
2,965772,Age,Age,a AD/CE,"[{'id': 1073135, 'name': 'age', 'semantic_uri'..."
3,965772,Date,Date,NaN,[]
4,965772,"Porites sp., Strontium/Calcium ratio",Porites sp. Sr/Ca,mmol/mol,"[{'id': 1055376, 'name': 'Porites', 'semantic_..."
5,965772,"Porites sp., δ18O",Porites sp. δ18O,‰ PDB,"[{'id': 1055376, 'name': 'Porites', 'semantic_..."
6,965772,"δ18O, seawater, reconstructed",δ18Osw,‰ PDB,"[{'id': 1073093, 'name': 'δ18O', 'semantic_uri..."
7,965772,Latitude,Latitude,deg,[]
8,965772,Longitude,Longitude,deg,[]
9,965772,Elevation,Elevation,m,[]


<div style="border-left: 4px solid #1f77b4; background: #f0f8ff; padding: 0.5em 1em; border-radius: 4px;">
  <strong>Note:</strong> Notice that the returned DataFrame looks quite different from its NOAA counterpart — the two repositories provide very different metadata fields. PyleoTUPS harmonizes column names (e.g., VariableName) where possible. 
</div>

## Working with Collections

Let's look at a study where multiple tables were needed to report results, leading to several datasets on the repository. In PANGAEA, these related datasets are grouped into a collection (also called a dataset publication series), each with its own DOI. 

For this tutorial, we use the collection by[Land and Reichle (2024)](https://doi.org/10.1594/PANGAEA.971943).

In [12]:
ds2 = pt.PangaeaDataset()

In [13]:
res2 = ds2.search_studies(study_ids = '971943')

[2026-04-24 10:22:59,639][INFO] - Registering Study 971943 via direct lookup.
[2026-04-24 10:23:00,708][INFO] - Retrived 1 studies
[2026-04-24 10:23:00,709][WARNING] - The search contains dataset(s) [971943] marked as collection. Refer to the 'CollectionMembers' column toidentify respective child datasets.


Note the warning as the results get downloaded. It informs us that one of the matching search criteria is a collection. 

Let's have a look at the summary:

In [14]:
df_summary = ds2.get_summary()
display(df_summary)

[2026-04-24 10:23:02,621][WARNING] - The search contains dataset(s) [971943] marked as collection. Refer to the 'CollectionMembers' column toidentify respective child datasets.


,StudyID,StudyName,EarliestYearBP,MostRecentYearBP,EarliestYearCE,MostRecentYearCE,StudyNotes,ScienceKeywords,Investigators,Publications,Sites,Funding,CollectionMembers
0,971943,Tree-ring width measurements of living Douglas...,None,None,None,None,Wood samples from Douglas fir trees were taken...,"[Central Europe, Pseudotsuga menziesii, Tree-r...","Land, Alexander, Reichle, Daniel","Land, Alexander; Reichle, Daniel (2024): Tree-...",[],[],"[972091, 972656, 972657, 972658, 972659, 97266..."


Let's have a closer look at the last column in our summary DataFrame, called `CollectionMembers`. The list contains all the datasets available as part of this collection:

In [15]:
print(df_summary['CollectionMembers'].iloc[0])

[972091, 972656, 972657, 972658, 972659, 972660, 972661, 972662, 972663, 972664, 972665, 972666, 972667, 972668, 972669, 972670, 972671, 972672, 972673, 972674, 972675, 972676, 972677, 972678, 972679, 972680, 972681, 972682, 972683, 972684, 972685, 972686, 972687, 972688, 972689, 972690, 972691, 972692, 972693, 972694, 972695, 972696, 972697, 972698, 972699, 972700, 972701, 972702]


What information can you get from the Collection? A few things. 

1. Geographical location

In [16]:
display(ds2.get_geo())

""


2. Funding information

Since this collection does not information about funding, the function will return an empty dataframe:

In [17]:
display(ds2.get_funding())

,StudyID,StudyName,FundingAgency,FundingGrant


3. Publication Information

In [18]:
bib, df_pub = ds2.get_publications()
display(df_pub)

,StudyID,StudyName,Author,Title,Journal,Year,Volume,Number,Pages,Type,DOI,URL,CitationKey
0,971943,Tree-ring width measurements of living Douglas...,"Land, Alexander; Reichle, Daniel",Tree-ring width measurements of living Douglas...,PANGAEA,2024,None,None,None,dataset,10.1594/PANGAEA.971943,https://doi.org/10.1594/PANGAEA.971943,"Land, Alexander; Reichle, Daniel (2024): Tree-..."


However, since our study is a collection, we cannot retrieve the data automatically:

In [19]:
df_data = ds2.get_data(study_id='971943')

[2026-04-24 10:23:13,090][WARNING] - Study 971943 is a collection dataset. Skipping.


But we can pass the member list from the summary DataFrame:

In [20]:
df_data = ds2.get_data(study_id=df_summary['CollectionMembers'].iloc[0])

[2026-04-24 10:23:15,021][INFO] - Study 972091 found as collection member. Registering child dataset.
[2026-04-24 10:23:16,664][INFO] - Study 972656 found as collection member. Registering child dataset.
[2026-04-24 10:23:19,452][INFO] - Study 972657 found as collection member. Registering child dataset.
[2026-04-24 10:23:21,109][INFO] - Study 972658 found as collection member. Registering child dataset.
[2026-04-24 10:23:22,760][INFO] - Study 972659 found as collection member. Registering child dataset.
[2026-04-24 10:23:24,404][INFO] - Study 972660 found as collection member. Registering child dataset.
[2026-04-24 10:23:26,076][INFO] - Study 972661 found as collection member. Registering child dataset.
[2026-04-24 10:23:27,785][INFO] - Study 972662 found as collection member. Registering child dataset.
[2026-04-24 10:23:29,403][INFO] - Study 972663 found as collection member. Registering child dataset.
[2026-04-24 10:23:31,083][INFO] - Study 972664 found as collection member. Registe

Let's have a look at the list:

In [21]:
len(df_data)

48

There are 48 data tables corresponding to the 48 Datasets in a Collection.

<div style="border-left: 4px solid #1f77b4; background: #f0f8ff; padding: 0.5em 1em; border-radius: 4px;">
  <strong>Why is this important:</strong> For now, you can pass a dataset ID directly. In practice, though, most users will query the data, and collections will be returned alongside their individual datasets. For example, consider a geographical query: if a collection contains datasets from across the globe, only some will match the query — but the collection itself will still appear in the results. You will then need to disambiguate based on the type of data you are interested in.  
</div>

## Summary

In this notebook, you learned how to access data and metadata for paleoclimate datasets stored on PANGAEA, and how this repository differs from NOAA NCEI. A subsequent notebook covers querying the PANGAEA database. 